In [ ]:
import torch
import urllib
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Adversarial Attack

An adversarial attack is a method in which you attempt to deceive a neural network by misclassifying an example with the smallest change to the input. An input that can deceive the network is known as an adversarial example. Adversarial examples are deliberately perturbed inputs aimed at deceiving the model, not interpreting it.

Adversarial attacks are a critical aspect of machine learning. In certain areas of research, misclassification is not tolerable. For instance, if someone were to place a small piece of tape on a stop sign, your self-driving car might misidentify it and drive through the intersection without issue

Various techniques have been developed to create adversarial examples. The majority of these approaches aim to minimize the distance between the adversarial example and the target instance, while simultaneously altering the prediction towards the intended (adversarial) outcome. Certain techniques require access to the model gradients, which limits their utility to gradient-based models like neural networks. Conversely, other methods solely necessitate access to the prediction function, rendering them model-agnostic.

In [ ]:
# Loading model
from torchvision.models.alexnet import AlexNet_Weights


model = torch.hub.load('pytorch/vision:v0.10.0', 'alexnet', weights=AlexNet_Weights.IMAGENET1K_V1)
_ = model.eval()

In [ ]:
# preprocessing
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

preprocess_pixels = transforms.Compose([
    transforms.Resize(256),  # resize
    transforms.CenterCrop(224),  # crop the center
    transforms.ToTensor(),  # convert to tensor in [0, 1]
])

normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])

# for a single instance
preprocess_instance = lambda x: preprocess(x).unsqueeze(0)
preprocess_pixels_instance = lambda x: preprocess_pixels(x).unsqueeze(0)

mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)

def normalize_batch(x):
    mean_ = mean.to(x.device, x.dtype)
    std_ = std.to(x.device, x.dtype)
    return (x - mean_) / std_

def denormalize_batch(x):
    mean_ = mean.to(x.device, x.dtype)
    std_ = std.to(x.device, x.dtype)
    return x * std_ + mean_

def get_predicted_labels(model, x_pixels):
    with torch.no_grad():
        logits = model(normalize_batch(x_pixels))
    return logits.argmax(dim=1)

def print_top5(prefix, logits):
    probabilities = torch.nn.functional.softmax(logits[0], dim=0)
    top5_prob, top5_catid = torch.topk(probabilities, 5)
    for prob, id_ in zip(top5_prob, top5_catid):
        print(f"[{prefix}] {categories[id_]} ({id_.item()}) with probability of {prob.item():.6f}")

In [ ]:
# download an example image
url, filename = ("https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg")

def download(url, filename):
    try: 
        urllib.URLopener().retrieve(url, filename)
    except: 
        urllib.request.urlretrieve(url, filename)
        
download(url, filename)

In [ ]:
input_image = Image.open(filename).convert("RGB")
input_pixels = preprocess_pixels_instance(input_image)
input_batch = normalize_batch(input_pixels)

In [ ]:
display(input_image)

In [ ]:
# downloading labels

download("https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt", "imagenet_classes.txt")
with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

In [ ]:
with torch.no_grad():
    output = model(input_batch)

probabilities = torch.nn.functional.softmax(output[0], dim=0)
top5_prob, top5_catid = torch.topk(probabilities, 5)

for i, id_ in enumerate(top5_catid):
    print(f"{categories[id_]} with probablity of {top5_prob[i]}")

In [ ]:
top5_catid

## Adversarial Attack (Szegedy-style Optimization)

Szegedy et al. (2014) introduced one of the first methods for generating adversarial examples by solving an optimization problem. The goal is to find a small perturbation $(r)$ such that the classifier’s prediction changes:

$$
\min_r \|r\|_2^2 + c \cdot \mathcal{L}(x + r)
$$

subject to the constraint that $$ x + r $$ remains a valid image.

### Untargeted Formulation

We want the model to **change its prediction**, to achieve this, we use a *margin-based loss* defined on the logits:

$$
\mathcal{L}_{\text{margin}}(x) =
\max\left( z_y(x) - \max_{j \ne y} z_j(x) + \kappa,\ 0 \right)
$$

where:
- $z_y(x)$ is the logit of the original (reference) class,
- $\max_{j \ne y} z_j(x)$ is the highest logit among all other classes,
- $\kappa \ge 0$ is a confidence margin.

This loss is zero only when another class has a higher logit than the original class by at least $\kappa$, meaning the decision boundary has been crossed.

### Final Objective

The optimization problem becomes:

$$
\min_r \|r\|_2^2 + c \cdot \max\left( z_y(x+r) - \max_{j \ne y} z_j(x+r) + \kappa,\ 0 \right)
$$

This formulation directly encourages **misclassification with minimal perturbation**.


### Algorithm

### Exmaples

In [ ]:
import torch
import torch.optim as optim
import tqdm
import matplotlib as mpl

def untargeted_margin_loss(logits, reference_label, kappa=0.0):
    B = logits.shape[0]
    true_logits = logits[torch.arange(B, device=logits.device), reference_label]
    tmp = logits.clone()
    tmp[torch.arange(B, device=logits.device), reference_label] = -1e9
    max_other = tmp.max(dim=1).values
    return torch.clamp(true_logits - max_other + kappa, min=0.0).mean()

def szegedy(image, model, iterations, lr=0.01, c=1.0, kappa=0.0, device="cpu"):
    model = model.to(device).eval()
    x_pixels = preprocess_pixels_instance(image).to(device)

    reference_label = get_predicted_labels(model, x_pixels)
    original_logits = model(normalize_batch(x_pixels))
    print_top5("Original Input", original_logits)

    r = torch.zeros_like(x_pixels, requires_grad=True)
    optimizer_adv = optim.Adam([r], lr=lr)

    for _ in tqdm.trange(iterations):
        adv_pixels = torch.clamp(x_pixels + r, 0, 1)
        logits = model(normalize_batch(adv_pixels))

        reg = (r.view(r.shape[0], -1) ** 2).sum(dim=1).mean()
        attack_loss = untargeted_margin_loss(logits, reference_label, kappa=kappa)
        loss_adv = reg + c * attack_loss

        optimizer_adv.zero_grad()
        loss_adv.backward()
        optimizer_adv.step()

    adv_pixels = torch.clamp(x_pixels + r, 0, 1).detach()
    with torch.no_grad():
        adv_logits = model(normalize_batch(adv_pixels))
    print_top5("Adversarial Input", adv_logits)

    img = x_pixels[0].detach().cpu().numpy().transpose(1, 2, 0)
    offset = (adv_pixels - x_pixels)[0].detach().cpu().numpy().transpose(1, 2, 0)
    changed = adv_pixels[0].detach().cpu().numpy().transpose(1, 2, 0)

    plt.figure(figsize=(10, 10))
    plt.subplot(1, 4, 1)
    plt.imshow(img.clip(0, 1))
    plt.xticks([])
    plt.yticks([])
    plt.title("Image")

    plt.subplot(1, 4, 2)
    plt.imshow(offset, cmap=mpl.colormaps["viridis"])
    plt.xticks([])
    plt.yticks([])
    plt.title("Noise")

    plt.subplot(1, 4, 3)
    plt.imshow(norm_pixels(offset), cmap=mpl.colormaps["viridis"])
    plt.xticks([])
    plt.yticks([])
    plt.title("Enhanced Noise")

    plt.subplot(1, 4, 4)
    plt.imshow(changed.clip(0, 1))
    plt.xticks([])
    plt.yticks([])
    plt.title("Adversarial")
    plt.show()

    return img, offset, changed

In [ ]:
files = (
    ("https://github.com/EliSchwartz/imagenet-sample-images/raw/master/n01440764_tench.JPEG", "tench.jpg"),
    ("https://github.com/EliSchwartz/imagenet-sample-images/raw/master/n01514668_cock.JPEG", "cock.jpg"),
    ("https://github.com/EliSchwartz/imagenet-sample-images/raw/master/n01807496_partridge.JPEG", "partridge.jpg"),
    ("https://github.com/EliSchwartz/imagenet-sample-images/raw/master/n01784675_centipede.JPEG", "centipede.jpg")
)

files = [(download(url, file), file)[1] for url, file in files]
images = [(file, Image.open(file)) for file in files]

In [ ]:
_, _, _ = szegedy(input_image, model, 300, lr=0.001, c=0.1, device="cuda")

In [ ]:
_, _, _ = szegedy(input_image, model, 300, lr=0.001, c=0.5, device="cuda")

In [ ]:
_, _, _ = szegedy(input_image, model, 300, lr=0.001, c=1, device="cuda")

In [ ]:
_, _, _ = szegedy(images[0][1], model, 300, lr=0.001, c=0.5, device="cuda")

In [ ]:
_, _, _ = szegedy(images[1][1], model, 300, lr=0.001, c=0.5, device="cuda")

In [ ]:
_, _, _ = szegedy(images[2][1], model, 300, lr=0.001, c=0.5, device="cuda")

In [ ]:
_, _, _ = szegedy(images[3][1], model, 300, lr=0.001, c=0.5, device="cuda")

In [ ]:
_, _, _ = szegedy(images[3][1], model, 300, lr=0.001, c=0.01, device="cuda")

## Fast Gradient Sign Method (Goodfellow et al., 2014)

Fast Gradient Sign Method (FGSM) uses the gradient of the loss with respect to the input to construct an adversarial image in a single step. For an attack, the perturbation is chosen to *increase* the loss of the reference label:

$$x_{\mathrm{adv}} = x + \epsilon \cdot \operatorname{sign}(\nabla_x J(\theta, x, y_{\mathrm{ref}})),$$

where $J$ is the loss, $x$ is the image, $y_{\mathrm{ref}}$ is the reference label (here the model's original prediction), and $\epsilon$ is the $\ell_\infty$ perturbation budget in **pixel space**.

In [ ]:
import torch.nn as nn
import matplotlib as mpl

norm_pixels = lambda x: (x - x.min()) / (x.max() - x.min() + 1e-12)

def fgsm(image, model, eps=8/255, device="cpu"):
    model = model.to(device).eval()
    loss = nn.CrossEntropyLoss()
    x_pixels = preprocess_pixels_instance(image).to(device)
    x_pixels.requires_grad_(True)

    reference_label = get_predicted_labels(model, x_pixels)
    original_logits = model(normalize_batch(x_pixels))
    print_top5("Original Input", original_logits)

    outputs = model(normalize_batch(x_pixels))
    loss_adv = loss(outputs, reference_label)

    grad = torch.autograd.grad(loss_adv, x_pixels, retain_graph=False, create_graph=False)[0]
    sign = grad.sign()
    adv_pixels = torch.clamp(x_pixels + eps * sign, 0, 1).detach()

    with torch.no_grad():
        adv_logits = model(normalize_batch(adv_pixels))
    print_top5("Adversarial Input", adv_logits)

    img = x_pixels[0].detach().cpu().numpy().transpose(1, 2, 0)
    offset = (adv_pixels - x_pixels)[0].detach().cpu().numpy().transpose(1, 2, 0)
    changed = adv_pixels[0].detach().cpu().numpy().transpose(1, 2, 0)

    plt.figure(figsize=(10, 10))
    plt.subplot(1, 4, 1)
    plt.imshow(img.clip(0, 1))
    plt.xticks([])
    plt.yticks([])
    plt.title("Image")
    plt.subplot(1, 4, 2)
    plt.imshow(offset, cmap=mpl.colormaps["viridis"])
    plt.xticks([])
    plt.yticks([])
    plt.title("Noise")
    plt.subplot(1, 4, 3)
    plt.imshow(norm_pixels(offset), cmap=mpl.colormaps["viridis"])
    plt.xticks([])
    plt.yticks([])
    plt.title("Noise Normalized")
    plt.subplot(1, 4, 4)
    plt.imshow(changed.clip(0, 1))
    plt.xticks([])
    plt.yticks([])
    plt.title("Adversarial Example")

    return img, offset, changed

In [ ]:
_, _, _ = fgsm(input_image, model, eps=8/255, device="cuda")

In [ ]:
_, _, _ = fgsm(images[0][1], model, eps=8/255, device="cuda")

In [ ]:
_, _, _ = fgsm(images[1][1], model, eps=8/255, device="cuda")

In [ ]:
_, _, _ = fgsm(images[2][1], model, eps=8/255, device="cuda")

In [ ]:
_, _, _ = fgsm(images[3][1], model, eps=8/255, device="cuda")

## Projected Gradient Descent (Madry et al., 2017)

PGD can be viewed as an iterative version of FGSM. For the $\ell_\infty$ attack, we approximately solve

$$\max_{\|\delta\|_\infty \leq \epsilon} J(\theta, x + \delta, y_{\mathrm{ref}}),$$

where $y_{\mathrm{ref}}$ is again the reference label (here the model's original prediction). After each gradient step, the adversarial example is projected back onto both the $\epsilon$-ball around the original image and the valid pixel range $[0, 1]$.

In [ ]:
import torch.nn as nn
import tqdm
import matplotlib as mpl

norm_pixels = lambda x: (x - x.min()) / (x.max() - x.min() + 1e-12)

def pgd(image, model, iterations, eps=8/255, alpha=2/255, device="cpu"):
    model = model.to(device).eval()
    loss = nn.CrossEntropyLoss()
    x_pixels = preprocess_pixels_instance(image).to(device)

    reference_label = get_predicted_labels(model, x_pixels)
    original_logits = model(normalize_batch(x_pixels))
    print_top5("Original Input", original_logits)

    adv_pixels = x_pixels.clone().detach()
    adv_pixels = adv_pixels + torch.empty_like(adv_pixels).uniform_(-eps, eps)
    adv_pixels = torch.max(torch.min(adv_pixels, x_pixels + eps), x_pixels - eps)
    adv_pixels = torch.clamp(adv_pixels, 0, 1).detach()

    for _ in tqdm.trange(iterations):
        adv_pixels.requires_grad_(True)
        outputs = model(normalize_batch(adv_pixels))
        cost = loss(outputs, reference_label)

        grad = torch.autograd.grad(cost, adv_pixels, retain_graph=False, create_graph=False)[0]

        adv_pixels = adv_pixels.detach() + alpha * grad.sign()
        delta = torch.clamp(adv_pixels - x_pixels, min=-eps, max=eps)
        adv_pixels = torch.clamp(x_pixels + delta, 0, 1).detach()

    with torch.no_grad():
        adv_logits = model(normalize_batch(adv_pixels))
    print_top5("Adversarial Input", adv_logits)

    img = x_pixels[0].detach().cpu().numpy().transpose(1, 2, 0)
    offset = (adv_pixels - x_pixels)[0].detach().cpu().numpy().transpose(1, 2, 0)
    changed = adv_pixels[0].detach().cpu().numpy().transpose(1, 2, 0)

    plt.figure(figsize=(10, 10))
    plt.subplot(1, 4, 1)
    plt.imshow(img.clip(0, 1))
    plt.xticks([])
    plt.yticks([])
    plt.title("Image")
    plt.subplot(1, 4, 2)
    plt.imshow(offset, cmap=mpl.colormaps["viridis"])
    plt.xticks([])
    plt.yticks([])
    plt.title("Noise")
    plt.subplot(1, 4, 3)
    plt.imshow(norm_pixels(offset), cmap=mpl.colormaps["viridis"])
    plt.xticks([])
    plt.yticks([])
    plt.title("Noise Normalized")
    plt.subplot(1, 4, 4)
    plt.imshow(changed.clip(0, 1))
    plt.xticks([])
    plt.yticks([])
    plt.title("Adversarial Example")

    return img, offset, changed

In [ ]:
_, _, _ = pgd(input_image, model, 300, device="cuda")

In [ ]:
_, _, _ = pgd(images[0][1], model, 300, device="cuda")

In [ ]:
_, _, _ = pgd(images[1][1], model, 300, device="cuda")

In [ ]:
_, _, _ = pgd(images[2][1], model, 300, device="cuda")

In [ ]:
_, _, _ = pgd(images[3][1], model, 300, device="cuda")

## OnePixel (Su et al., 2019)

The 1-pixel attack looks for a modified example $x'$ which comes close to the original image $x$, but changes the prediction to an adversarial outcome. However, the definition of closeness differs: Only a single pixel may change ($\ell_0$). The main difference from the earlier approaches is that we are going to find a solution with differential evolution. Each candidate solution encodes a pixel modification and is represented by a vector of five elements: the x- and y-coordinates and the red, green and blue (RGB) values. A candaidate can be generated as:

$$x_i(g+1)=x_{r1}(g) + F\cdot(x_{r2}(g) - x_{r3}(g))$$

where each $x_i$ is an element of a candidate solution (either x-coordinate, y-coordinate, red, green or blue), $g$ is the current generation, $F$ is a scaling parameter and $r1$, $r2$ and $r3$ are different random numbers. Each new child candidate solution is in turn a pixel with the five attributes for location and color and each of those attributes is a mixture of three random parent pixels.

In [ ]:
!pip install torchattacks

In [ ]:
import torchattacks

torch.backends.cudnn.deterministic = True

def onepixel(model, img, pixels, device="cuda"):
    model = model.to(device).eval()
    atk = torchattacks.OnePixel(model, pixels=pixels)
    atk.set_normalization_used(mean=IMAGENET_MEAN, std=IMAGENET_STD)

    x_pixels = preprocess_pixels_instance(img).to(device)
    reference_label = get_predicted_labels(model, x_pixels)

    with torch.no_grad():
        output = model(normalize_batch(x_pixels))
    print_top5("Original Input", output)

    adv_norm = atk(normalize_batch(x_pixels), reference_label)
    adv_pixels = denormalize_batch(adv_norm).detach().clamp(0, 1)

    with torch.no_grad():
        adv_output = model(normalize_batch(adv_pixels))
    print_top5("Adversarial Input", adv_output)

    rev_img = x_pixels[0].detach().cpu().numpy().transpose(1, 2, 0)
    denormed_adv_img = adv_pixels[0].detach().cpu().numpy().transpose(1, 2, 0)

    plt.figure(figsize=(10, 10))
    plt.subplot(1, 2, 1)
    plt.imshow(rev_img.clip(0, 1))
    plt.xticks([])
    plt.yticks([])
    plt.title("Image")
    plt.subplot(1, 2, 2)
    plt.imshow(denormed_adv_img.clip(0, 1))
    plt.xticks([])
    plt.yticks([])
    plt.title("Adversarial Example")

    return denormed_adv_img

In [ ]:
_ = onepixel(model, input_image, 10)

In [ ]:
_ = onepixel(model, input_image, 50)

In [ ]:
_ = onepixel(model, input_image, 100)

In [ ]:
_ = onepixel(model, images[0][1], 10)

In [ ]:
_ = onepixel(model, images[1][1], 10)

In [ ]:
_ = onepixel(model, images[2][1], 10)

In [ ]:
_ = onepixel(model, images[3][1], 10)

# Adversarial Training

Adversarial training is a technique in machine learning where a model is trained to be robust against adversarial examples. Adversarial examples are inputs that have been deliberately designed to cause a machine learning model to make an incorrect prediction. The goal of adversarial training is to improve the model's generalization performance by exposing it to adversarial examples during training.

In adversarial training, the model is trained on a mixture of normal examples and adversarial examples generated from the normal examples. The adversarial examples are created by perturbing the normal examples in a way that is designed to maximize the model's prediction error. By training on both normal and adversarial examples, the model learns to be more robust and resilient to these types of attacks.

Adversarial training is particularly useful in applications where the security and robustness of the model are critical, such as in computer vision for autonomous driving, malware detection, or fraud detection in finance.